In [3]:
import pandas as pd
import numpy as np
import os

# Load all 3 raw files
cdc = pd.read_csv("data/raw/cdc_filtered.csv")
sci = pd.read_csv("data/raw/sci_within_county.csv")
census = pd.read_csv("data/raw/census_acs.csv")

print("CDC shape:", cdc.shape)
print("SCI shape:", sci.shape)
print("Census shape:", census.shape)


CDC shape: (11828, 6)
SCI shape: (3221, 2)
Census shape: (3221, 9)


In [4]:
# Pivot so each measure becomes its own column
cdc_wide = cdc.pivot_table(
    index=['county', 'state', 'fips', 'population'],
    columns='measure',
    values='value'
).reset_index()

# Flatten column names
cdc_wide.columns.name = None
cdc_wide.columns = [
    'county', 'state', 'fips', 'population',
    'depression_rate', 'mental_distress_rate'
]

print(f"CDC wide shape: {cdc_wide.shape}")
print(cdc_wide.head(3))

CDC wide shape: (2956, 6)
      county           state   fips  population  depression_rate  \
0  Abbeville  South Carolina  45001       24434            24.00   
1     Acadia       Louisiana  22001       56489            31.00   
2   Accomack        Virginia  51001       33239            22.55   

   mental_distress_rate  
0                 18.55  
1                 22.20  
2                 18.15  


In [5]:
# Standardize FIPS to 5-digit string in all 3 datasets
cdc_wide['fips'] = cdc_wide['fips'].astype(str).str.zfill(5)
sci['fips']      = sci['fips'].astype(str).str.zfill(5)
census['fips']   = census['fips'].astype(str).str.zfill(5)

# Verify they all look the same
print("CDC FIPS sample:    ", cdc_wide['fips'].head(3).tolist())
print("SCI FIPS sample:    ", sci['fips'].head(3).tolist())
print("Census FIPS sample: ", census['fips'].head(3).tolist())

CDC FIPS sample:     ['45001', '22001', '51001']
SCI FIPS sample:     ['01001', '01003', '01005']
Census FIPS sample:  ['01001', '01003', '01005']


In [6]:
# Merge CDC + Census first
master = pd.merge(cdc_wide, census, on='fips', how='inner')
print(f"After CDC + Census merge: {master.shape}")

# Then merge SCI
master = pd.merge(master, sci, on='fips', how='inner')
print(f"After adding SCI: {master.shape}")

print("\nAll columns in master dataframe:")
for col in master.columns:
    print(f"  {col}")
    

After CDC + Census merge: (2947, 14)
After adding SCI: (2947, 15)

All columns in master dataframe:
  county
  state
  fips
  population
  depression_rate
  mental_distress_rate
  name
  median_income
  median_age
  poverty_pop
  broadband_pop
  total_pop
  state_code
  county_code
  sci


In [7]:
# 'total_pop' already exists from CDC as 'population' — duplicate, drop it
# 'name' is the long Census name like "Autauga County, Alabama" — we already have 'county'
master = master.drop(columns=['total_pop', 'name'], errors='ignore')

# Rename to shorter, cleaner names for easier typing later
master = master.rename(columns={
    'median_income':   'income',
    'median_age':      'age',
    'poverty_pop':     'poverty',
    'broadband_pop':   'broadband'
})

print(f"Final shape: {master.shape}")
print(master.head(3))

Final shape: (2947, 13)
      county           state   fips  population  depression_rate  \
0  Abbeville  South Carolina  45001       24434            24.00   
1     Acadia       Louisiana  22001       56489            31.00   
2   Accomack        Virginia  51001       33239            22.55   

   mental_distress_rate  income   age  poverty  broadband  state_code  \
0                 18.55   45710  44.3     4072       6867          45   
1                 22.20   42368  36.8    13209      16787          22   
2                 18.15   50601  47.1     5593      10899          51   

   county_code       sci  
0            1  0.364204  
1            1  0.435191  
2            1  0.410062  


In [8]:
print("Missing values per column:")
print(master.isnull().sum())
print(f"\nTotal rows before dropping nulls: {len(master)}")

# Our target variables — if these are missing, the row is useless, remove it
master = master.dropna(subset=['depression_rate', 'mental_distress_rate'])

# For other columns, fill gaps with the median value of that column
# Median is better than mean here because income data is skewed by very rich counties
for col in ['income', 'age', 'poverty', 'broadband']:
    median_val = master[col].median()
    master[col] = master[col].fillna(median_val)

print(f"Total rows after cleaning: {len(master)}")

Missing values per column:
county                  0
state                   0
fips                    0
population              0
depression_rate         0
mental_distress_rate    0
income                  0
age                     0
poverty                 0
broadband               0
state_code              0
county_code             0
sci                     0
dtype: int64

Total rows before dropping nulls: 2947
Total rows after cleaning: 2947


In [9]:
# Poverty as % of total county population — now comparable across counties
master['poverty_rate'] = (master['poverty'] / master['population']) * 100

# Broadband access as % of population — measures digital connectivity
master['broadband_rate'] = (master['broadband'] / master['population']) * 100

# Split counties into 4 income groups — Q1=poorest 25%, Q4=richest 25%
# This is how we'll later ask "does SCI help poor counties differently than rich ones?"
master['income_quartile'] = pd.qcut(
    master['income'],
    q=4,
    labels=['Q1 Low', 'Q2 Mid-Low', 'Q3 Mid-High', 'Q4 High']
)

print("New features added:")
print(master[['county', 'state', 'poverty_rate',
              'broadband_rate', 'income_quartile']].head(5))

New features added:
      county           state  poverty_rate  broadband_rate income_quartile
0  Abbeville  South Carolina     16.665302       28.104281          Q1 Low
1     Acadia       Louisiana     23.383314       29.717290          Q1 Low
2   Accomack        Virginia     16.826619       32.789795      Q2 Mid-Low
3        Ada           Idaho      8.300980       32.094657         Q4 High
4      Adair            Iowa     11.327649       32.805522     Q3 Mid-High


In [10]:
master.to_csv("data/processed/master.csv", index=False)
print(f"Master dataframe saved!")
print(f"Shape: {master.shape}")
print(f"\nFinal columns:")
for i, col in enumerate(master.columns, 1):
    print(f"  {i:2}. {col}")

Master dataframe saved!
Shape: (2947, 16)

Final columns:
   1. county
   2. state
   3. fips
   4. population
   5. depression_rate
   6. mental_distress_rate
   7. income
   8. age
   9. poverty
  10. broadband
  11. state_code
  12. county_code
  13. sci
  14. poverty_rate
  15. broadband_rate
  16. income_quartile


In [11]:
print("=== MASTER DATAFRAME SUMMARY ===\n")
print(f"Total counties: {len(master):,}")
print(f"States covered: {master['state'].nunique()}")
print(f"\nDepression rate:      min={master['depression_rate'].min():.1f}%  "
      f"max={master['depression_rate'].max():.1f}%  "
      f"avg={master['depression_rate'].mean():.1f}%")
print(f"Mental distress rate: min={master['mental_distress_rate'].min():.1f}%  "
      f"max={master['mental_distress_rate'].max():.1f}%  "
      f"avg={master['mental_distress_rate'].mean():.1f}%")
print(f"Median income:        ${master['income'].median():,.0f}")
print(f"SCI range:            {master['sci'].min():.3f} to {master['sci'].max():.3f}")
print(f"\nSample counties:")
print(master[['county', 'state', 'depression_rate',
              'mental_distress_rate', 'sci', 'income']].head(5))

=== MASTER DATAFRAME SUMMARY ===

Total counties: 2,947
States covered: 48

Depression rate:      min=12.3%  max=35.2%  avg=23.5%
Mental distress rate: min=12.4%  max=25.4%  avg=17.9%
Median income:        $56,128
SCI range:            0.122 to 0.679

Sample counties:
      county           state  depression_rate  mental_distress_rate       sci  \
0  Abbeville  South Carolina            24.00                 18.55  0.364204   
1     Acadia       Louisiana            31.00                 22.20  0.435191   
2   Accomack        Virginia            22.55                 18.15  0.410062   
3        Ada           Idaho            25.25                 15.50  0.510118   
4      Adair            Iowa            21.05                 17.25  0.433395   

   income  
0   45710  
1   42368  
2   50601  
3   75115  
4   57944  


In [12]:
master.to_csv("data/processed/master.csv", index=False)
print(f"Master dataframe saved!")
print(f"Shape: {master.shape}")
print(f"\nFinal columns:")
for i, col in enumerate(master.columns, 1):
    print(f"  {i:2}. {col}")

Master dataframe saved!
Shape: (2947, 16)

Final columns:
   1. county
   2. state
   3. fips
   4. population
   5. depression_rate
   6. mental_distress_rate
   7. income
   8. age
   9. poverty
  10. broadband
  11. state_code
  12. county_code
  13. sci
  14. poverty_rate
  15. broadband_rate
  16. income_quartile


In [13]:
print("=== MASTER DATAFRAME SUMMARY ===\n")
print(f"Total counties: {len(master):,}")
print(f"States covered: {master['state'].nunique()}")
print(f"\nDepression rate:      min={master['depression_rate'].min():.1f}%  "
      f"max={master['depression_rate'].max():.1f}%  "
      f"avg={master['depression_rate'].mean():.1f}%")
print(f"Mental distress rate: min={master['mental_distress_rate'].min():.1f}%  "
      f"max={master['mental_distress_rate'].max():.1f}%  "
      f"avg={master['mental_distress_rate'].mean():.1f}%")
print(f"Median income:        ${master['income'].median():,.0f}")
print(f"SCI range:            {master['sci'].min():.3f} to {master['sci'].max():.3f}")
print(f"\nSample counties:")
print(master[['county', 'state', 'depression_rate',
              'mental_distress_rate', 'sci', 'income']].head(5))

=== MASTER DATAFRAME SUMMARY ===

Total counties: 2,947
States covered: 48

Depression rate:      min=12.3%  max=35.2%  avg=23.5%
Mental distress rate: min=12.4%  max=25.4%  avg=17.9%
Median income:        $56,128
SCI range:            0.122 to 0.679

Sample counties:
      county           state  depression_rate  mental_distress_rate       sci  \
0  Abbeville  South Carolina            24.00                 18.55  0.364204   
1     Acadia       Louisiana            31.00                 22.20  0.435191   
2   Accomack        Virginia            22.55                 18.15  0.410062   
3        Ada           Idaho            25.25                 15.50  0.510118   
4      Adair            Iowa            21.05                 17.25  0.433395   

   income  
0   45710  
1   42368  
2   50601  
3   75115  
4   57944  


In [14]:
master.head(5)

,county,state,fips,population,depression_rate,mental_distress_rate,income,age,poverty,broadband,state_code,county_code,sci,poverty_rate,broadband_rate,income_quartile
0,Abbeville,South Carolina,45001,24434,24.00,18.55,45710,44.3,4072,6867,45,1,0.364204,16.665302,28.104281,Q1 Low
1,Acadia,Louisiana,22001,56489,31.00,22.20,42368,36.8,13209,16787,22,1,0.435191,23.383314,29.717290,Q1 Low
2,Accomack,Virginia,51001,33239,22.55,18.15,50601,47.1,5593,10899,51,1,0.410062,16.826619,32.789795,Q2 Mid-Low
3,Ada,Idaho,16001,524673,25.25,15.50,75115,37.7,43553,168392,16,1,0.510118,8.300980,32.094657,Q4 High
4,Adair,Iowa,19001,7389,21.05,17.25,57944,43.4,837,2424,19,1,0.433395,11.327649,32.805522,Q3 Mid-High
